## Monte Carlo Policy Evaluation

In [ ]:
import gymnasium as gym
import numpy as np
from collections import defaultdict

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Create Blackjack environment
env = gym.make('Blackjack-v1')

In [ ]:
def sample_policy(observation):
    """A simple policy for Blackjack: stick if sum is >= 20, hit otherwise"""
    score, dealer_score, usable_ace = observation
    return 0 if score >= 20 else 1  # 0: stick, 1: hit

def generate_episode(env, policy):
    """Generate an episode using the given policy"""
    episode = []
    observation, _ = env.reset()
    done = False
    while not done:
        action = policy(observation)
        next_observation, reward, terminated, truncated, _ = env.step(action)
        episode.append((observation, action, reward))
        observation = next_observation
        done = terminated or truncated
    return episode

In [ ]:
def first_visit_mc_prediction(env, policy, num_episodes=100000):
    returns = defaultdict(list)
    value_function = defaultdict(float)

    for _ in range(num_episodes):
        episode = generate_episode(env, policy)
        state_action_pairs = set()
        for t, (state, action, reward) in enumerate(episode):
            if (state, action) not in state_action_pairs:
                state_action_pairs.add((state, action))
                G = sum(r for _, _, r in episode[t:])
                returns[(state, action)].append(G)
                value_function[(state, action)] = np.mean(returns[(state, action)])

    return value_function

# Run first-visit Monte Carlo prediction
first_visit_value_function = first_visit_mc_prediction(env, sample_policy)

In [ ]:
def every_visit_mc_prediction(env, policy, num_episodes=100000):
    returns = defaultdict(list)
    value_function = defaultdict(float)

    for _ in range(num_episodes):
        episode = generate_episode(env, policy)
        for t, (state, action, reward) in enumerate(episode):
            G = sum(r for _, _, r in episode[t:])
            returns[(state, action)].append(G)
            value_function[(state, action)] = np.mean(returns[(state, action)])

    return value_function

# Run every-visit Monte Carlo prediction
every_visit_value_function = every_visit_mc_prediction(env, sample_policy)

In [ ]:
def plot_value_function(value_function, title, filename, cmap='YlGnBu'):
    # Prepare data for plotting
    player_sum = range(12, 22)
    dealer_show = range(1, 11)
    
    stick_values = np.zeros((len(player_sum), len(dealer_show), 2))
    hit_values = np.zeros((len(player_sum), len(dealer_show), 2))
    
    for (state, action), value in value_function.items():
        player_total, dealer_card, usable_ace = state
        if 12 <= player_total <= 21 and 1 <= dealer_card <= 10:
            if action == 0:  # Stick
                stick_values[player_total-12, dealer_card-1, usable_ace] = value
            else:  # Hit
                hit_values[player_total-12, dealer_card-1, usable_ace] = value
    
    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 15))
    fig.suptitle(title, fontsize=16)
    
    # Plot heatmaps
    for i, ace in enumerate(['No Usable Ace', 'Usable Ace']):
        sns.heatmap(stick_values[:, :, i], ax=axes[i, 0], cmap=cmap, 
                    xticklabels=dealer_show, yticklabels=player_sum)
        axes[i, 0].set_title(f'Stick - {ace}')
        axes[i, 0].set_xlabel('Dealer Showing')
        axes[i, 0].set_ylabel('Player Sum')
        
        sns.heatmap(hit_values[:, :, i], ax=axes[i, 1], cmap=cmap, 
                    xticklabels=dealer_show, yticklabels=player_sum)
        axes[i, 1].set_title(f'Hit - {ace}')
        axes[i, 1].set_xlabel('Dealer Showing')
        axes[i, 1].set_ylabel('Player Sum')
    
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
plot_value_function(first_visit_value_function, "First-Visit MC Value Function", 'bjk_firstvaluemc.png')
plot_value_function(first_visit_value_function, "First-Visit MC Value Function", 'bjk_firstvaluemc_bw.png', 'Greys')

In [ ]:
plot_value_function(first_visit_value_function, "Every-Visit MC Value Function", 'bjk_everyvaluemc.png')
plot_value_function(first_visit_value_function, "Every-Visit MC Value Function", 'bjk_everyvaluemc_bw.png', 'Greys')